# R-Tree, Virtual Tables And Extensions

## Quick notes
### What this covers
- Virtual tables: tables that look normal in SQL but are backed by code (R-Tree, FTS5, DBSTAT, GEOPOLY, `json_each`, `pragma_*`).
- The R-Tree spatial index: store bounding boxes, ask "what overlaps this box?" fast.
- A "stores near me" query: bounding-box prefilter, then exact distance with SQL math.
- Emulating `generate_series` with a recursive CTE, math functions, and why loadable extensions are usually off.
- A feature map of what SQLite lacks compared with PostgreSQL.
### Key points
- `CREATE VIRTUAL TABLE t USING rtree(id, minX, maxX, minY, maxY)` makes a spatial index. Each row is a box (a point is a box with min = max).
- R-Tree answers box questions (inside, overlaps) quickly. It does not do "nearest" by itself, so use a box prefilter plus a distance formula.
- Table-valued functions (`json_each`, `pragma_table_info`, `pragma_index_list`) are virtual tables you call like a function in `FROM`.
- `WITH RECURSIVE` builds number and date series when `generate_series` is not available.
- SQLite is flexible about types and small on syntax. Know the gaps before moving to PostgreSQL.
### Gotchas (the part worth revising twice)
- R-Tree stores coordinates as 32-bit floats, so values come back slightly changed (18.5167 comes back as 18.51669692993164). Keep exact numbers in a normal table.
- R-Tree columns come in min/max pairs, and min greater than max is an error. Ids must be unique.
- A box prefilter is a rough filter: points in the corners of the box can be farther than the radius. Always finish with the exact distance.
- Virtual tables cannot have ordinary indexes, triggers or foreign keys, and most are slow if you scan them without the special search form.
- R-Tree, GEOPOLY, FTS5, DBSTAT and the math functions are compile-time options. This runtime has them, but another build may not. Guard code that must run anywhere (see section 08).
- Loadable extensions are disabled by default for security. Turning them on means running native code from a file.
- `generate_series` is an extension, not built in. Use a recursive CTE.
### When it matters
- Maps, "find within a rectangle" lookups, collision checks, calendar-slot overlap.
- Inspecting your own database from SQL (`pragma_table_info`, `dbstat`) without leaving SQL.
- Planning a migration to another database engine.
### One-line recap example
- `SELECT id FROM place_idx WHERE minX >= 73.8 AND maxX <= 73.9 AND minY >= 18.4 AND maxY <= 18.6;` means "every place whose box sits fully inside this rectangle, found through the spatial index".

## 01. Seed data: shops with coordinates
- A real table keeps the exact data: name, latitude, longitude.
- The spatial index will hold only the boxes and the ids. The real table stays the source of truth.
- Coordinates are near Pune, India (latitude about 18.5, longitude about 73.8).

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE shops (
    shop_id INTEGER PRIMARY KEY, name TEXT NOT NULL, kind TEXT NOT NULL,
    lat REAL NOT NULL, lon REAL NOT NULL
);
INSERT INTO shops VALUES
    (1,'Camp Books','books',18.5167,73.8800),
    (2,'Deccan Cafe','cafe',18.5170,73.8410),
    (3,'Kothrud Mart','grocery',18.5074,73.8077),
    (4,'Hinjewadi Tech','electronics',18.5912,73.7389),
    (5,'Koregaon Brew','cafe',18.5362,73.8937),
    (6,'Baner Books','books',18.5590,73.7868),
    (7,'Hadapsar Fresh','grocery',18.5089,73.9259),
    (8,'Katraj Cafe','cafe',18.4575,73.8677);
SELECT * FROM shops ORDER BY shop_id;

shop_id | name           | kind        | lat     | lon    
--------+----------------+-------------+---------+--------
1       | Camp Books     | books       | 18.5167 | 73.88  
2       | Deccan Cafe    | cafe        | 18.517  | 73.841 
3       | Kothrud Mart   | grocery     | 18.5074 | 73.8077
4       | Hinjewadi Tech | electronics | 18.5912 | 73.7389
5       | Koregaon Brew  | cafe        | 18.5362 | 73.8937
6       | Baner Books    | books       | 18.559  | 73.7868
7       | Hadapsar Fresh | grocery     | 18.5089 | 73.9259
8       | Katraj Cafe    | cafe        | 18.4575 | 73.8677
(8 rows)



## 02. Virtual tables: the idea
- A virtual table looks like a table but SQLite asks a module (a piece of code) for the rows, instead of reading a B-tree page.
- `CREATE VIRTUAL TABLE name USING module(args)` makes one that stores data (rtree, fts5, geopoly).
- Some modules are read-only views of something else: `dbstat` (storage use), `json_each` (a JSON value as rows), `pragma_*` (settings as rows).
- `pragma_module_list` lists the modules in this build. Check it before relying on one.

In [3]:
%%sql
SELECT name FROM pragma_module_list
WHERE name IN ('rtree','geopoly','fts5','dbstat','json_each','json_tree')
ORDER BY name;

name     
---------
dbstat   
fts5     
geopoly  
json_each
json_tree
rtree    
(6 rows)



## 03. Create an R-Tree
- Columns: an integer id first, then pairs of `min`/`max` for each dimension (1 to 5 dimensions, so 3 to 11 columns).
- A point is a box where min equals max.
- Fill it from the real table. The id links back to `shops`.

In [4]:
%%sql
CREATE VIRTUAL TABLE shop_idx USING rtree(shop_id, min_lat, max_lat, min_lon, max_lon);
INSERT INTO shop_idx SELECT shop_id, lat, lat, lon, lon FROM shops;
SELECT COUNT(*) AS indexed FROM shop_idx;

indexed
-------
8      
(1 row)



## 04. Bounding-box queries
- "Inside the box": the shop's min is at or above the box min, and its max is at or below the box max.
- "Overlaps the box": its min is at or below the box max, and its max is at or above the box min. For points both mean the same.
- The index is only used when the `WHERE` compares the R-Tree columns with plain constants. Wrapping a column in a function loses it.
- Join back to `shops` for names.
- In the plan below, `VIRTUAL TABLE INDEX 2:...` means the R-Tree search was used. The code after the colon lists the tests handed to the R-Tree: a letter (`D` = `>=`, `B` = `<=`, `A` = `=`, `C` = `<`, `E` = `>`) plus the number of the coordinate column (0 = first coordinate). So `D0B1D2B3` is the four range tests of the query below.

In [5]:
%%sql
SELECT s.shop_id, s.name
FROM shop_idx i JOIN shops s USING (shop_id)
WHERE i.min_lat >= 18.50 AND i.max_lat <= 18.54
  AND i.min_lon >= 73.80 AND i.max_lon <= 73.90
ORDER BY s.shop_id;

shop_id | name         
--------+--------------
1       | Camp Books   
2       | Deccan Cafe  
3       | Kothrud Mart 
5       | Koregaon Brew
(4 rows)



In [6]:
%%sql
EXPLAIN QUERY PLAN
SELECT shop_id FROM shop_idx
WHERE min_lat >= 18.50 AND max_lat <= 18.54 AND min_lon >= 73.80 AND max_lon <= 73.90;

id | parent | notused | detail                                      
---+--------+---------+---------------------------------------------
2  | 0      | 0       | SCAN shop_idx VIRTUAL TABLE INDEX 2:D0B1D2B3
(1 row)



## 05. The float gotcha, and an auxiliary column
- The index keeps 32-bit floats, so what comes back is close but not identical to what you stored.
- Boxes are rounded outward on insert, so a search never misses a true match. It may rarely include a point just outside the edge.
- Fix: keep exact values in the base table (we did) and recheck there.
- Auxiliary columns (`+name`, SQLite 3.24+) store extra data next to the box without indexing it. Handy for a label, but you cannot search on them quickly.

In [7]:
%%sql
SELECT shop_id, lat, min_lat, ROUND(min_lat - lat, 9) AS drift
FROM shop_idx JOIN shops USING (shop_id)
WHERE shop_id IN (1, 2) ORDER BY shop_id;

shop_id | lat     | min_lat            | drift     
--------+---------+--------------------+-----------
1       | 18.5167 | 18.51669692993164  | -3.07e-06 
2       | 18.517  | 18.516998291015625 | -1.709e-06
(2 rows)



In [8]:
%%sql
CREATE VIRTUAL TABLE zone_idx USING rtree(zone_id, min_x, max_x, min_y, max_y, +zone_name TEXT);
INSERT INTO zone_idx VALUES
    (1, 0, 10, 0, 10, 'Warehouse'),
    (2, 8, 15, 5, 12, 'Loading dock'),
    (3, 20, 30, 0, 5, 'Office'),
    (4, 14, 22, 3, 8, 'Walkway'),
    (5, 40, 50, 40, 50, 'Parking');
SELECT zone_id, zone_name, min_x, max_x, min_y, max_y FROM zone_idx ORDER BY zone_id;

zone_id | zone_name    | min_x | max_x | min_y | max_y
--------+--------------+-------+-------+-------+------
1       | Warehouse    | 0.0   | 10.0  | 0.0   | 10.0 
2       | Loading dock | 8.0   | 15.0  | 5.0   | 12.0 
3       | Office       | 20.0  | 30.0  | 0.0   | 5.0  
4       | Walkway      | 14.0  | 22.0  | 3.0   | 8.0  
5       | Parking      | 40.0  | 50.0  | 40.0  | 50.0 
(5 rows)



## 06. Overlap detection
- Two boxes overlap when they overlap on every axis: `a.min <= b.max AND a.max >= b.min`, for x and for y.
- Joining the R-Tree to itself uses the index for the inner lookups.
- `a.zone_id < b.zone_id` keeps each pair once and drops "a zone overlaps itself".
- Same idea checks double-booked rooms (one axis is time) or colliding sprites.

In [9]:
%%sql
SELECT a.zone_name AS first_zone, b.zone_name AS second_zone
FROM zone_idx a JOIN zone_idx b
  ON b.zone_id > a.zone_id
 AND b.min_x <= a.max_x AND b.max_x >= a.min_x
 AND b.min_y <= a.max_y AND b.max_y >= a.min_y
ORDER BY a.zone_id, b.zone_id;

first_zone   | second_zone 
-------------+-------------
Warehouse    | Loading dock
Loading dock | Walkway     
Office       | Walkway     
(3 rows)



## 07. Nearest-ish queries: box, then distance
- R-Tree answers boxes, not "closest". Pattern: take a box around the point, then order the few candidates by distance.
- For a flat map a quick distance is `(dx*dx + dy*dy)` (no square root needed just to sort).
- If the box is too small and returns nothing, widen it and try again (do that in application code).
- Here: the 3 shops nearest to the point 18.52, 73.85 inside a box of plus or minus 0.1 degree.

In [10]:
%%sql
SELECT s.name,
       ROUND((s.lat - 18.52) * (s.lat - 18.52) + (s.lon - 73.85) * (s.lon - 73.85), 5) AS dist_sq
FROM shop_idx i JOIN shops s USING (shop_id)
WHERE i.min_lat BETWEEN 18.42 AND 18.62
  AND i.min_lon BETWEEN 73.75 AND 73.95
ORDER BY dist_sq, s.shop_id
LIMIT 3;

name         | dist_sq
-------------+--------
Deccan Cafe  | 9e-05  
Camp Books   | 0.00091
Kothrud Mart | 0.00195
(3 rows)



## 08. Maths functions: guard first
- SQLite 3.35+ can include math functions (`sqrt`, `pow`, `ln`, `log10`, `sin`, `cos`, `asin`, `radians`, `pi()`...) if built with `ENABLE_MATH_FUNCTIONS`.
- Many builds have them, some (older distro packages) do not. The cell below checks.
- The haversine example in the next section needs `radians`, `sin`, `cos`, `asin`, `sqrt`.

In [11]:
needed = ["sqrt", "pow", "ln", "log10", "sin", "cos", "asin", "radians", "pi"]
have = {r[0] for r in con.execute("SELECT name FROM pragma_function_list")}
print({f: (f in have) for f in needed})
print("math ready:", all(f in have for f in needed))

{'sqrt': True, 'pow': True, 'ln': True, 'log10': True, 'sin': True, 'cos': True, 'asin': True, 'radians': True, 'pi': True}
math ready: True


In [12]:
%%sql
SELECT sqrt(144) AS root, pow(2, 10) AS power, round(ln(exp(1)), 6) AS ln_e, log10(1000) AS log10_1000,
       round(sin(radians(30)), 6) AS sin30, round(pi(), 6) AS pi, floor(-2.5) AS floor_, ceil(-2.5) AS ceil_;

root | power  | ln_e | log10_1000 | sin30 | pi       | floor_ | ceil_
-----+--------+------+------------+-------+----------+--------+------
12.0 | 1024.0 | 1.0  | 3.0        | 0.5   | 3.141593 | -3.0   | -2.0 
(1 row)



## 09. Stores near me: box prefilter + exact haversine
- Step 1, rough: convert the radius to degrees and use the R-Tree to get candidates. 1 degree of latitude is about 111 km. A degree of longitude is about `111 * cos(latitude)` km.
- Step 2, exact: compute the great-circle distance (haversine) on the few candidates and keep the ones within the radius.
- The box removes the far-away shops cheaply. The exact formula removes the corner cases. With only 8 shops all 4 candidates here are really within 6 km, so the exact step removes none; with more data, shops in the box corners would be dropped.
- Query point here: 18.5204, 73.8567, radius 6 km. Parameters live in a CTE so you change them in one place.

In [13]:
%%sql
WITH me(lat, lon, km) AS (SELECT 18.5204, 73.8567, 6.0),
cand AS (
    SELECT s.shop_id, s.name, s.kind, s.lat, s.lon, me.lat AS mlat, me.lon AS mlon, me.km
    FROM me
    JOIN shop_idx i
      ON i.min_lat BETWEEN me.lat - me.km / 111.0 AND me.lat + me.km / 111.0
     AND i.min_lon BETWEEN me.lon - me.km / (111.0 * cos(radians(me.lat)))
                       AND me.lon + me.km / (111.0 * cos(radians(me.lat)))
    JOIN shops s ON s.shop_id = i.shop_id
)
SELECT name, kind,
       ROUND(2 * 6371 * asin(sqrt(
           pow(sin(radians(lat - mlat) / 2), 2) +
           cos(radians(mlat)) * cos(radians(lat)) * pow(sin(radians(lon - mlon) / 2), 2)
       )), 2) AS km_away
FROM cand
WHERE 2 * 6371 * asin(sqrt(
           pow(sin(radians(lat - mlat) / 2), 2) +
           cos(radians(mlat)) * cos(radians(lat)) * pow(sin(radians(lon - mlon) / 2), 2)
       )) <= km
ORDER BY km_away, shop_id;

name          | kind    | km_away
--------------+---------+--------
Deccan Cafe   | cafe    | 1.7    
Camp Books    | books   | 2.49   
Koregaon Brew | cafe    | 4.28   
Kothrud Mart  | grocery | 5.36   
(4 rows)



## 10. Keeping the index in sync
- An R-Tree is a normal writable table: `INSERT`, `UPDATE`, `DELETE` all work.
- It is not linked to `shops` automatically. If you change a shop, change its box too. Triggers (see `05-DDL-And-Integrity`) can do this for you.
- Moving a shop: update both places, in one transaction.

In [14]:
%%sql
CREATE TRIGGER shops_ai AFTER INSERT ON shops BEGIN
    INSERT INTO shop_idx VALUES (NEW.shop_id, NEW.lat, NEW.lat, NEW.lon, NEW.lon);
END;
CREATE TRIGGER shops_au AFTER UPDATE OF lat, lon ON shops BEGIN
    UPDATE shop_idx SET min_lat = NEW.lat, max_lat = NEW.lat, min_lon = NEW.lon, max_lon = NEW.lon
    WHERE shop_id = NEW.shop_id;
END;
CREATE TRIGGER shops_ad AFTER DELETE ON shops BEGIN
    DELETE FROM shop_idx WHERE shop_id = OLD.shop_id;
END;
INSERT INTO shops VALUES (9, 'Pashan Pantry', 'grocery', 18.5400, 73.7900);
UPDATE shops SET lat = 18.5410 WHERE shop_id = 9;
SELECT i.shop_id, s.name, ROUND(i.min_lat, 4) AS idx_lat FROM shop_idx i JOIN shops s USING (shop_id) WHERE shop_id = 9;
DELETE FROM shops WHERE shop_id = 9;
SELECT (SELECT COUNT(*) FROM shops) AS shops, (SELECT COUNT(*) FROM shop_idx) AS indexed;

shop_id | name          | idx_lat
--------+---------------+--------
9       | Pashan Pantry | 18.541 
(1 row)

shops | indexed
------+--------
8     | 8      
(1 row)



## 11. R-Tree rules that bite
- Min above max is rejected.
- Re-using an id fails.
- The index has no foreign keys and no real type checks (a NULL coordinate is silently stored as 0.0), so keep it in step with the base table.

In [15]:
%%sql expect_error
INSERT INTO shop_idx VALUES (100, 19.0, 18.0, 73.0, 74.0);

ERROR: rtree constraint failed: shop_idx.(min_lat<=max_lat)



In [16]:
%%sql expect_error
INSERT INTO shop_idx VALUES (1, 18.0, 18.0, 73.0, 73.0);

ERROR: UNIQUE constraint failed: shop_idx.shop_id



## 12. GEOPOLY: polygons (guarded)
- GEOPOLY is a sibling module that stores 2D polygons and has functions like `geopoly_area`, `geopoly_contains_point`, `geopoly_overlap`.
- It is a compile option too, so this cell checks `pragma_module_list` and skips quietly if missing.
- A polygon is a JSON list of `[x, y]` points. The last point repeats the first, or SQLite closes it for you.
- The cell is Python only so the guard can wrap the SQL.

In [17]:
mods = {r[0] for r in con.execute("SELECT name FROM pragma_module_list")}
if "geopoly" not in mods:
    print("GEOPOLY not built in here, skipping")
else:
    con.execute("CREATE VIRTUAL TABLE areas USING geopoly(label)")
    con.execute("INSERT INTO areas(_shape, label) VALUES ('[[0,0],[4,0],[4,4],[0,4],[0,0]]', 'square')")
    con.execute("INSERT INTO areas(_shape, label) VALUES ('[[3,3],[8,3],[8,8],[3,8],[3,3]]', 'big-square')")
    con.execute("INSERT INTO areas(_shape, label) VALUES ('[[10,10],[12,10],[11,12],[10,10]]', 'triangle')")
    print("areas:", con.execute("SELECT label, geopoly_area(_shape) FROM areas ORDER BY label").fetchall())
    print("contains (2,2):", con.execute(
        "SELECT label FROM areas WHERE geopoly_contains_point(_shape, 2, 2) ORDER BY label").fetchall())
    print("overlap with [[2,2],[5,2],[5,5],[2,5]]:", con.execute(
        "SELECT label FROM areas WHERE geopoly_overlap(_shape, '[[2,2],[5,2],[5,5],[2,5],[2,2]]') ORDER BY label").fetchall())

areas: [('big-square', 25.0), ('square', 16.0), ('triangle', 2.0)]
contains (2,2): [('square',)]
overlap with [[2,2],[5,2],[5,5],[2,5]]: [('big-square',), ('square',)]


## 13. Table-valued functions: pragma_* and json_each
- Pragmas can be read as tables: `pragma_table_info('shops')`, `pragma_index_list(...)`, `pragma_foreign_key_list(...)`. That lets you filter and join them like any table.
- `json_each(json)` turns a JSON array or object into rows (`key`, `value`, `type`). `json_tree` walks nested JSON.
- Both are virtual tables you call with arguments, written in `FROM`.
- For JSON itself, see `08-SQLite-Specials/01-JSON-In-SQLite`.

In [18]:
%%sql
SELECT cid, name, type, "notnull" AS not_null, pk FROM pragma_table_info('shops') ORDER BY cid;

cid | name    | type    | not_null | pk
----+---------+---------+----------+---
0   | shop_id | INTEGER | 0        | 1 
1   | name    | TEXT    | 1        | 0 
2   | kind    | TEXT    | 1        | 0 
3   | lat     | REAL    | 1        | 0 
4   | lon     | REAL    | 1        | 0 
(5 rows)



In [19]:
%%sql
SELECT m.name AS tbl, COUNT(*) AS columns
FROM sqlite_master m JOIN pragma_table_info(m.name) p
WHERE m.type = 'table' AND m.name IN ('shops', 'zone_idx')
GROUP BY m.name ORDER BY m.name;

tbl      | columns
---------+--------
shops    | 5      
zone_idx | 6      
(2 rows)



In [20]:
%%sql
SELECT key, value, type FROM json_each('{"open":"09:00","close":"21:00","days":["Mon","Tue"],"wifi":true}')
ORDER BY key;

key   | value         | type 
------+---------------+------
close | 21:00         | text 
days  | ["Mon","Tue"] | array
open  | 09:00         | text 
wifi  | 1             | true 
(4 rows)



In [21]:
%%sql
CREATE TABLE tagged (id INTEGER PRIMARY KEY, tags TEXT);
INSERT INTO tagged VALUES (1, '["cafe","wifi"]'), (2, '["books"]'), (3, '["cafe","quiet","books"]');
SELECT t.id, j.value AS tag FROM tagged t, json_each(t.tags) j ORDER BY t.id, j.key;

id | tag  
---+------
1  | cafe 
1  | wifi 
2  | books
3  | cafe 
3  | quiet
3  | books
(6 rows)



## 14. DBSTAT and FTS5: two more virtual tables
- `dbstat` shows how pages are used per table and index. Good for "which table is big?". It needs `ENABLE_DBSTAT_VTAB`.
- `fts5` is full-text search. `MATCH` finds words, `rank` orders by relevance. It needs `ENABLE_FTS5`. In this notebook it is only a glimpse.
- `dbstat` needs no `CREATE`: it is always available when the build includes it. An `fts5` (or `rtree`) table you create appears in `sqlite_master`, with shadow tables behind the scenes. Do not edit those by hand.

In [22]:
%%sql
SELECT name, COUNT(*) AS pages, SUM(ncell) AS cells
FROM dbstat WHERE name IN ('shops', 'tagged') GROUP BY name ORDER BY name;

name   | pages | cells
-------+-------+------
shops  | 1     | 8    
tagged | 1     | 3    
(2 rows)



In [23]:
%%sql
CREATE VIRTUAL TABLE blurb USING fts5(title, body);
INSERT INTO blurb VALUES
    ('Pune cafes', 'quiet cafes with fast wifi near the river'),
    ('Book shops', 'second hand books and quiet reading corners'),
    ('Tech stores', 'laptops phones and wifi routers');
SELECT title FROM blurb WHERE blurb MATCH 'quiet' ORDER BY title;
SELECT title FROM blurb WHERE blurb MATCH 'wifi NOT quiet' ORDER BY title;

title     
----------
Book shops
Pune cafes
(2 rows)

title      
-----------
Tech stores
(1 row)



## 15. generate_series without the extension
- `generate_series(start, stop, step)` exists in PostgreSQL and in SQLite's optional `series` extension, but it is not built into the standard SQLite library.
- A recursive CTE does the same job. Stop it with a `WHERE` so it ends.
- Handy for calendars, filling gaps in a report, and test data. For more on recursion, see `02-Querying`.

In [24]:
%%sql
WITH RECURSIVE nums(n) AS (SELECT 1 UNION ALL SELECT n + 1 FROM nums WHERE n < 5)
SELECT n, n * n AS square FROM nums ORDER BY n;

n | square
--+-------
1 | 1     
2 | 4     
3 | 9     
4 | 16    
5 | 25    
(5 rows)



In [25]:
%%sql
WITH RECURSIVE days(d) AS (
    SELECT '2024-02-27' UNION ALL SELECT date(d, '+1 day') FROM days WHERE d < '2024-03-02'
)
SELECT d, strftime('%w', d) AS weekday_no FROM days ORDER BY d;

d          | weekday_no
-----------+-----------
2024-02-27 | 2         
2024-02-28 | 3         
2024-02-29 | 4         
2024-03-01 | 5         
2024-03-02 | 6         
(5 rows)



In [26]:
%%sql
WITH RECURSIVE nums(n) AS (SELECT 0 UNION ALL SELECT n + 5 FROM nums WHERE n < 20)
SELECT group_concat(n, ',') AS step5 FROM (SELECT n FROM nums ORDER BY n);

step5       
------------
0,5,10,15,20
(1 row)



## 16. Loadable extensions: the concept
- A loadable extension is a compiled library (`.so`, `.dylib`, `.dll`) that adds functions, collations or virtual tables, loaded with `SELECT load_extension('path')` or the `.load` shell command.
- It is off by default because loading a library means running arbitrary native code. A SQL injection that can call `load_extension` would take over the machine.
- Python keeps it off: you must call `con.enable_load_extension(True)` first, and many Python builds do not even include the method.
- Turn it on only for a trusted file, for the shortest time, then switch it off again. No demo here on purpose.
- Alternative with no native code: register Python functions with `create_function` (see `04-SQLite-With-Python`).

In [27]:
print("has enable_load_extension:", hasattr(con, "enable_load_extension"))
try:
    con.execute("SELECT load_extension('nothing')")
except Exception as e:
    print(type(e).__name__, "-", e)

has enable_load_extension: True
OperationalError - not authorized


## 17. User-defined functions: a recap
- Scalar, aggregate and window functions written in Python are the safe extension route: see `04-SQLite-With-Python` (sections on `create_function`, `create_aggregate`, `create_window_function`).
- They run per row in Python, so they are slower than built-ins. Use them for logic SQL cannot express, not for plain arithmetic.
- They live on one connection. The next connection does not know them.

## 18. Feature map: what SQLite lacks, side by side
- A cheat sheet before moving to another engine. Full comparison: the dialects notebook (planned as `10-Dialects`).
- Rule of thumb: SQLite is a library inside your program with simple rules, PostgreSQL is a server with many more features.

In [28]:
%%sql
WITH t(feature, sqlite, postgresql) AS (VALUES
    ('Types',            'flexible: a column type is a hint',            'strict types, many kinds'),
    ('Dates',            'no date type; TEXT, REAL or INTEGER',           'DATE, TIMESTAMP, INTERVAL types'),
    ('ALTER COLUMN',     'not supported; rebuild the table',              'supported'),
    ('Stored procedures','none; use triggers, app code, UDFs',            'functions and procedures'),
    ('RIGHT / FULL JOIN','from version 3.39 (2022)',                      'always'),
    ('Concurrency',      'one writer at a time per database file',        'many writers (MVCC)'),
    ('Users and grants', 'none; file permissions decide',                 'roles and GRANT'),
    ('generate_series',  'not built in; recursive CTE',                   'built in'),
    ('Arrays, ranges',   'none; JSON text instead',                       'native arrays and range types')
)
SELECT * FROM t ORDER BY feature;

feature           | sqlite                                 | postgresql                     
------------------+----------------------------------------+--------------------------------
ALTER COLUMN      | not supported; rebuild the table       | supported                      
Arrays, ranges    | none; JSON text instead                | native arrays and range types  
Concurrency       | one writer at a time per database file | many writers (MVCC)            
Dates             | no date type; TEXT, REAL or INTEGER    | DATE, TIMESTAMP, INTERVAL types
RIGHT / FULL JOIN | from version 3.39 (2022)               | always                         
Stored procedures | none; use triggers, app code, UDFs     | functions and procedures       
Types             | flexible: a column type is a hint      | strict types, many kinds       
Users and grants  | none; file permissions decide          | roles and GRANT                
generate_series   | not built in; recursive CTE            | built in 

## 19. Feature map in action
- Flexible typing: an INTEGER column accepts text that does not look like a number. A `STRICT` table (3.37+) turns that off.
- No `ALTER COLUMN`: you get a syntax error. The usual fix is create-copy-drop-rename (see `05-DDL-And-Integrity`).
- No date type: `date()` returns TEXT.
- RIGHT JOIN works here only because this build is 3.39 or newer.

In [29]:
%%sql
CREATE TABLE loose (n INTEGER);
INSERT INTO loose VALUES (42), ('42'), ('hello'), (4.5);
SELECT n, typeof(n) AS stored_as FROM loose ORDER BY rowid;

n     | stored_as
------+----------
42    | integer  
42    | integer  
hello | text     
4.5   | real     
(4 rows)



In [30]:
%%sql expect_error
CREATE TABLE tight (n INTEGER) STRICT;
INSERT INTO tight VALUES ('hello');

ERROR: cannot store TEXT value in INTEGER column tight.n



In [31]:
%%sql expect_error
ALTER TABLE loose ALTER COLUMN n TYPE TEXT;

ERROR: near "ALTER": syntax error



In [32]:
%%sql
SELECT typeof(date('2024-03-09')) AS date_type, date('2024-03-09', '+1 month') AS next_month;

date_type | next_month
----------+-----------
text      | 2024-04-09
(1 row)



In [33]:
%%sql
SELECT s.name, i.shop_id
FROM shop_idx i RIGHT JOIN shops s ON s.shop_id = i.shop_id
WHERE s.kind = 'books' ORDER BY s.shop_id;

name        | shop_id
------------+--------
Camp Books  | 1      
Baner Books | 6      
(2 rows)

